# Computer Vision Assignment 1 — Mango Grading Line

**Purpose:** one Colab-ready notebook containing executable engineering calculations, synthetic demonstrations, and analysis scaffolds for Tasks 1–10.

**Important:** the supplied brief requires experimental results from mango images. This notebook does not invent those results; synthetic demonstrations are explicitly labelled as such.


In [ ]:
# ============================================================
# SETUP — RUN THIS CELL FIRST
# ============================================================
import os, math, time, json, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cv2
except Exception:
    !pip -q install opencv-python-headless
    import cv2

from sklearn.metrics import cohen_kappa_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

BASE = Path("/content/mango_cv_assignment")
DATA = BASE / "data"
OUT = BASE / "outputs"
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

BELT_SPEED_MPS = 0.5
FOV_X_MM, FOV_Y_MM = 400, 300
MAX_FRUITS_PER_S = 8
DECISION_LIMIT_S = 1.2
EJECTOR_DISTANCE_M = 0.6
MIN_DEFECT_MM = 2.0
SIZE_TOL_MM = 2.0
MAX_DEFECT_MISS = 0.02
CAMERA_HEIGHT_MM = 500
STORAGE_BYTES = 2e12
SHIFT_HOURS = 10

print("OpenCV:", cv2.__version__)
print("Workspace:", BASE)


## Task 1 — Grading approaches and agreement


In [ ]:
# Task 1
manual_shift_capacity = MAX_FRUITS_PER_S * 3600 * SHIFT_HOURS
print("10-hour demand at 8 fruits/s:", manual_shift_capacity, "fruit decisions")
demo_a=[0,1,1,0,2,2,1,0,1,2]
demo_b=[0,1,0,0,2,2,1,0,1,2]
print("Synthetic Cohen kappa:", round(float(cohen_kappa_score(demo_a,demo_b)),4))


## Task 2 — Pipeline depth


In [ ]:
# Task 2
def make_synthetic_mango(seed=0,h=256,w=384):
    rng=np.random.default_rng(seed)
    img=np.zeros((h,w,3),dtype=np.uint8); img[:]=(45,85,125)
    cv2.ellipse(img,(w//2,h//2),(int(w*.27),int(h*.38)),0,0,360,(70,150,190),-1)
    for _ in range(3):
        x=rng.integers(w//2-int(w*.13),w//2+int(w*.13))
        y=rng.integers(h//2-int(h*.19),h//2+int(h*.19))
        cv2.circle(img,(int(x),int(y)),int(rng.integers(3,8)),(20,30,35),-1)
    return img
def low_level(img):
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); blur=cv2.GaussianBlur(gray,(5,5),0)
    _,bw=cv2.threshold(blur,70,255,cv2.THRESH_BINARY_INV)
    return {"defective":int(np.count_nonzero(bw))>1500,"dark_pixels":int(np.count_nonzero(bw))}
def mid_level(img):
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    mask=cv2.inRange(hsv,np.array([0,50,0]),np.array([179,255,230]))
    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    areas=[cv2.contourArea(c) for c in contours if cv2.contourArea(c)>50]
    return {"regions":len(areas),"largest_area":max(areas) if areas else 0}
def high_level(img):
    m=mid_level(img); return {"grade":"reject" if m["largest_area"]>15000 else "accept",**m}
img=make_synthetic_mango(1)
print(low_level(img)); print(mid_level(img)); print(high_level(img))
plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()


## Task 3 — RGB/BGR, HSV and Lab


In [ ]:
# Task 3
img=make_synthetic_mango(2); hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV); lab=cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
print("shape:",img.shape,"dtype:",img.dtype)
M,N,k=img.shape
print("Uncompressed bits:",M*N*k*8,"bytes:",M*N*k)
a=np.array([250],dtype=np.uint8); b=np.array([10],dtype=np.uint8)
print("NumPy uint8 +:",(a+b)[0],"cv2.add:",cv2.add(a,b)[0])
plt.hist(hsv[...,0].ravel(),bins=36); plt.xlabel("HSV H"); plt.show()


## Task 4 — Connectivity and distance measures


In [ ]:
# Task 4
pattern=np.zeros((40,60),dtype=np.uint8); pattern[10,10]=255; pattern[11,11]=255; pattern[12,12]=255; pattern[20:24,30:34]=255
for conn in (4,8):
    n,labels,stats,cent=cv2.connectedComponentsWithStats(pattern,connectivity=conn)
    print(conn,"-connectivity components:",n-1)
p=np.array([10,20]); q=np.array([14,23]); d=np.abs(p-q)
print("Euclidean:",np.linalg.norm(p-q),"City-block:",d.sum(),"Chessboard:",d.max())
plt.imshow(pattern,cmap="gray"); plt.axis("off"); plt.show()


## Task 5 — Acquisition, motion blur and line-scan rate


In [ ]:
# Task 5
for exposure_ms in [0.1,1.0,4.0]:
    blur_mm=BELT_SPEED_MPS*exposure_ms/1000*1000
    print(exposure_ms,"ms ->",blur_mm,"mm motion blur")
line_pitch_mm=FOV_X_MM/2448
line_rate=(BELT_SPEED_MPS*1000)/line_pitch_mm
print("Line pitch:",line_pitch_mm,"mm; base line rate:",line_rate,"lines/s; 3x:",3*line_rate)


## Task 6 — Spatial sampling, frame rate, data rate and storage


In [ ]:
# Task 6
rows=[]
for w,h in [(640,480),(1280,1024),(2448,2048)]:
    mmpp=FOV_X_MM/w; px2=2/mmpp; bytes_frame=w*h*3
    rate=bytes_frame*MAX_FRUITS_PER_S/1e6
    store=bytes_frame*MAX_FRUITS_PER_S*60*60*24*7/1e9
    rows.append([f"{w}x{h}",mmpp,px2,rate,store])
display(pd.DataFrame(rows,columns=["Resolution","mm/pixel","pixels/2mm","Raw MB/s @ 8fps","Raw GB/7d"]))
print("Approx. frame rate from 190 mm available travel:",BELT_SPEED_MPS/0.190,"fps")


## Task 7 — Quantization and bruise contrast


In [ ]:
# Task 7
def quantize_uint8(x,bits):
    levels=2**bits; step=256/levels
    return np.clip(np.floor(x/step)*step,0,255).astype(np.uint8)
ramp=np.tile(np.arange(256,dtype=np.uint8),(80,1)); ramp[30:50,100:120]=154
for bits in [8,6,5,4]:
    q=quantize_uint8(ramp,bits)
    print(bits,"-bit example contrast:",abs(int(q[35,100])-int(q[20,100])))


## Task 8 — Projection, focal length and illumination correction


In [ ]:
# Task 8
sensor_width_mm=6.4
f_mm=sensor_width_mm*CAMERA_HEIGHT_MM/FOV_X_MM
print("Example focal length for 6.4 mm sensor width:",f_mm,"mm")
Z=CAMERA_HEIGHT_MM; h=90
ratio=Z/(Z-h)
print("Perspective scale ratio at 90 mm fruit height:",ratio)
print("Approx. error if ignored:",(ratio-1)*100,"%")
H,W=240,320; yy,xx=np.mgrid[0:H,0:W]
illum=0.7+0.3*(xx/W); scene=np.full((H,W),120.0); scene[90:130,140:180]=80.0
observed=scene*illum; corrected=observed/illum*np.mean(illum)
fig,ax=plt.subplots(1,3,figsize=(12,3))
for a,t in zip(ax,[scene,observed,corrected]): a.imshow(t,cmap="gray"); a.axis("off")
plt.show()


## Task 9 — Integrated front-end calculations


In [ ]:
# Task 9
timing_ms={"Acquisition":10,"Transfer":8,"Low-level":20,"Mid-level":25,"High-level":35,"Decision+I/O":10}
total_ms=sum(timing_ms.values()); print("Example total:",total_ms,"ms; against 1.2 s:",total_ms/1000,"s")
display(pd.DataFrame([
["Budget","Low cost","Lighting/blur sensitivity"],
["Balanced","Global shutter + strobe + calibration","Moderate integration"],
["Premium","High resolution / telecentric / line scan","Higher cost and storage"],
["Smart camera","Onboard acceleration","Vendor/model maintenance"]
],columns=["Configuration","Strength","Main risk"]))


## Task 10 — Adaptation to 0.5 mm defects, 1.0 m/s and NIR


In [ ]:
# Task 10
NEW_SPEED=1.0; NEW_DEFECT_MM=0.5; NEW_FPS=16
target_px=4; mmpp=NEW_DEFECT_MM/target_px; required_width=FOV_X_MM/mmpp
print("Target mm/pixel:",mmpp,"required width:",required_width,"pixels")
for exposure_ms in [0.1,0.5,1.0]:
    print(exposure_ms,"ms ->",NEW_SPEED*exposure_ms,"mm blur")
w=math.ceil(required_width); h=2048; bytes_frame=w*h*3
print("Illustrative raw MB/s:",bytes_frame*NEW_FPS/1e6)
print("Illustrative raw TB/7d:",bytes_frame*NEW_FPS*60*60*24*7/1e12)
display(pd.DataFrame([
["A — Retain + ROI","Lowest change","May fail 0.5 mm/NIR requirement"],
["B — Modify sampling/acquisition","Better spatial/temporal control","No NIR information"],
["C — RGB + NIR redesign","Addresses NIR-only defects","Highest integration complexity"]
],columns=["Strategy","Advantage","Limitation"]))


## Run all
Use **Runtime → Run all**. Real-dataset sections must be populated with the required mango images to produce empirical results.
